# **🔬 Clean & Jerk Dataset — Individual Set Inspector**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import DEFAULT_CLEAN_DATASET_DIR, load_summary_tables, load_set_h5

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

### 🎯 Select & Load a Set (`.h5`)

In [ ]:
# @title 🎯 Load Set HDF5 { display-mode: "form" }
import pandas as pd

set_id = "set_0001" # @param {type:"string"}
set_data = load_set_h5(dataset_dir, set_id)

print(f"✅ Loaded {set_id}: {set_data['accel'].shape[0]} samples ({set_data['time_s'][-1]:.2f}s) | {len(set_data['sensors'])} IMUs")
df_set_meta = pd.DataFrame([set_data["metadata"]])
df_part_meta = pd.DataFrame([set_data["participant_metadata"]])
display_table(df_set_meta, clean_columns=False)
display_table(df_part_meta, clean_columns=False)

### 🩺 26-IMU Sensor Status

In [ ]:
# @title 🩺 26-IMU Sensor Status
from src.explorer import plot_set_sensor_status

plot_set_sensor_status(set_data).show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### ⚠️ Set Fault Log (`/fault_log`)

In [ ]:
# @title ⚠️ Set Fault Log
df_set_faults = set_data["fault_log_df"]
display_table(df_set_faults, clean_columns=False)

### ⏱️ Phase Markers — Consensus & Per-Annotator

In [ ]:
# @title ⏱️ Phase Markers
df_consensus = set_data["consensus_df"]
df_annotators = set_data["annotators_df"]
print(f"Consensus markers: {len(df_consensus)} rows | Individual annotator markers: {len(df_annotators)} rows")
display_table(df_consensus, clean_columns=False)
display_table(df_annotators, clean_columns=False)

### 📈 IMU Signal + Phase Markers Overlay

In [ ]:
# @title 📈 IMU Signal + Phase Markers Overlay { display-mode: "form" }
from src.explorer import plot_set_imu_with_markers

sensor_code = "RKP" # @param ["RKP","RHD","REP","RSD","LKP","LHD","LEP","LSD","LHP","RHP","LS","RWD","RWP","RED","LWD","LWP","LED","LAD","LAP","LKD","LSP","RSP","US","RAD","RAP","RKD"]
lift_id = 0 # @param {type:"integer"}  # 0 = All Lifts

fig_imu = plot_set_imu_with_markers(
    set_data,
    sensor_code=sensor_code,
    lift_id=None if lift_id == 0 else lift_id,
)
fig_imu.show(config={"toImageButtonOptions": {"format": "png", "scale": 3}})

### 📋 Full IMU DataFrame (`/imu_data`)

In [ ]:
# @title 📋 Full IMU DataFrame
df_imu = set_data["imu_df"]
display_table(df_imu.head(200), clean_columns=False)

### 🎥 Set Videos (`/videos/set_XXXX_*.mp4`)

In [ ]:
# @title 🎥 Preview Set Video { display-mode: "form" }
from IPython.display import Video, display

camera_view = "front" # @param ["front", "side"]
vid_path = os.path.join(dataset_dir, "videos", f"{set_id}_{camera_view}.mp4")
if os.path.exists(vid_path):
    print(f"🎬 {vid_path} ({os.path.getsize(vid_path)/1e6:.1f} MB)")
    display(Video(vid_path, width=640, embed=False))
else:
    print(f"ℹ️ No {camera_view} video found at {vid_path}")